# 08 · Kiểm tra và kết luận

Kiểm tra tính nhất quán các tệp đã lưu và mốc dự đoán lớp đa số; không dùng test để chọn lại cấu hình.

Notebook chạy độc lập từ thư mục gốc hoặc `notebooks/`; cài `requirements-demo.txt` trước.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT/'student_success').exists(): ROOT = ROOT.parent
assert (ROOT/'student_success').exists(), 'Chạy từ thư mục dự án hoặc notebooks/'
sys.path.insert(0, str(ROOT))
from student_success.data import CATEGORICAL, NUMERIC, LABELS, SEED, split
DATA = ROOT/'data/raw/data.csv'
assert DATA.exists(), 'Chạy python scripts/download_data.py'
df = pd.read_csv(DATA, sep=';'); df.columns = df.columns.str.strip()
train_idx, test_idx = split(df)
train = df.loc[train_idx]
R = ROOT/'data/experiment'


In [2]:
import subprocess
check=subprocess.run([sys.executable,str(ROOT/'scripts/verify_results.py')],cwd=ROOT,text=True,capture_output=True)
print(check.stdout)
if check.returncode: raise RuntimeError(check.stderr)

PASS: four models, identical 885 test rows, train-only folds, predictions, confusion matrices and metrics



In [3]:
from sklearn.metrics import accuracy_score, f1_score, recall_score
majority=train.Target.mode().iloc[0]; y=df.loc[test_idx,'Target']; baseline=[majority]*len(y)
print('Lớp đa số:',majority,'| Accuracy:',round(accuracy_score(y,baseline),4),'| Macro-F1:',round(f1_score(y,baseline,average='macro'),4),'| Recall Dropout:',round(recall_score(y,baseline,labels=['Dropout'],average='macro',zero_division=0),4))

Lớp đa số: Graduate | Accuracy: 0.4994 | Macro-F1: 0.2221 | Recall Dropout: 0.0


Một lần chia trên một cơ sở không chứng minh mô hình tổng quát hóa. Cần kiểm định độc lập trước khi ứng dụng thực tế.